# Ray checkpointing example

This notebook runs a **Ray Train** checkpointing demo on **Red Hat OpenShift AI** using the CodeFlare SDK:

- **Red Hat build of Kueue** must be configured in your cluster (ResourceFlavor → ClusterQueue → **LocalQueue** in your namespace, and `kueue.openshift.io/managed=true` on the namespace). See the OpenShift *AI workloads* documentation for [Red Hat build of Kueue](https://docs.redhat.com/en/documentation/openshift_container_platform/4.21/html/ai_workloads/red-hat-build-of-kueue).
- Submit a **RayJob** with a **managed Ray cluster** (`ManagedClusterConfig`) so KubeRay lifecycles the cluster with the job (`shutdownAfterJobFinishes`). The RayJob is labeled for your **LocalQueue** via `local_queue` (example: `"default"`).
- Configure **AWS credentials** for the S3 bucket used by Ray Train checkpoints.
- **Monitor** training in the Ray dashboard (**Jobs** tab), then **suspend and resume** the RayJob (`job.stop()` / `job.resubmit()`) to verify training **resumes from S3** after a simulated interruption.

Training script: `train_with_checkpoints.py` in this directory (same source as the CodeFlare SDK guided demo).

## Import required libraries

In [ ]:
from codeflare_sdk import RayJob, ManagedClusterConfig, set_api_client, get_cluster
from kube_authkit import AuthConfig, get_k8s_client
import time

## Authenticate to your OpenShift cluster

In [ ]:
import urllib3

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# Authenticate to your Kubernetes/OpenShift cluster using kube-authkit

# Option 1: Auto-detect credentials (kubeconfig or in-cluster service account)
# NOTE: In RHOAI Workbenches the workbench service account may not have Ray RBAC
# permissions. Use Option 2 (token) unless your admin has granted SA permissions
# (see RHOAIENG-46748). Auto-detect works if you have a local kubeconfig.
# auth_config = AuthConfig(method="auto")

# Option 2 (Recommended for RHOAI Workbenches): Token-based authentication
# Get your token with: oc whoami -t  (or from the OpenShift console → Copy login command)
auth_config = AuthConfig(
    method="openshift",
    k8s_api_host="https://api.example.com:6443",
    token="sha256~XXXXX",  # oc whoami -t
)

# Option 3: OIDC authentication (for BYOIDC-enabled clusters)
# auth_config = AuthConfig(
#     method="oidc",
#     k8s_api_host="https://api.example.com:6443",
#     oidc_issuer="https://your-oidc-provider.com",
#     client_id="your-client-id",
#     use_device_flow=True,  # Interactive device flow for notebook environments
# )

api_client = get_k8s_client(config=auth_config)
# Set to False for self-signed / dev API certificates (optional).
api_client.configuration.verify_ssl = False
set_api_client(api_client)

NAMESPACE = "your-namespace"  # Data Science Project where LocalQueue + RayJob run
JOB_NAME = "checkpointing-job"
# Must match metadata.name of a LocalQueue in NAMESPACE (create per OpenShift Kueue docs).
LOCAL_QUEUE = "default"

## Red Hat build of Kueue (required before submit)

Configure **ResourceFlavor** → **ClusterQueue** → **LocalQueue** in your project namespace, and label the namespace so Kueue manages workloads there. Official OpenShift 4.21 *AI workloads* docs:

- [Configuring a resource flavor](https://docs.redhat.com/en/documentation/openshift_container_platform/4.21/html/ai_workloads/red-hat-build-of-kueue#configuring-resourceflavors_configuring-quotas)
- [Configuring a cluster queue](https://docs.redhat.com/en/documentation/openshift_container_platform/4.21/html/ai_workloads/red-hat-build-of-kueue#configuring-clusterqueues_configuring-quotas)
- [Configuring a local queue](https://docs.redhat.com/en/documentation/openshift_container_platform/4.21/html/ai_workloads/red-hat-build-of-kueue#configuring-localqueues_configuring-quotas) — the `LocalQueue` `metadata.name` must match **`LOCAL_QUEUE`** above (e.g. `default`).
- [Labeling namespaces to allow Red Hat build of Kueue to manage jobs](https://docs.redhat.com/en/documentation/openshift_container_platform/4.21/html/ai_workloads/red-hat-build-of-kueue#labeling-namespaces-to-allow-red-hat-build-of-kueue-to-manage-jobs_managing-jobs-and-workloads): `oc label namespace <namespace> kueue.openshift.io/managed=true`

After submit, the RayJob may show **Suspended** until Kueue **admits** a `Workload` — use `oc get workloads.kueue.x-k8s.io -n $NAMESPACE` if needed. That is **not** the same as the manual suspend used later for the checkpoint demo.

In [ ]:
# Optional: verify Kueue objects exist (cluster admin / user with read access)
# !oc get resourceflavor.kueue.x-k8s.io
# !oc get clusterqueue.kueue.x-k8s.io
# !oc get localqueue.kueue.x-k8s.io -n $NAMESPACE

print(f"Namespace: {NAMESPACE!r}, RayJob name: {JOB_NAME!r}, LocalQueue: {LOCAL_QUEUE!r}")

## Set your AWS credentials

In [ ]:
# Set your AWS credentials
# WARNING: Do not commit credentials to version control. For production,
# use OpenShift AI Data Connections or OpenShift Secrets instead.
AWS_CREDENTIALS = {
    "AWS_ACCESS_KEY_ID": "your-access-key",
    "AWS_SECRET_ACCESS_KEY": "your-secret-key",
    "AWS_DEFAULT_REGION": "us-east-1",  # e.g. "us-east-1"
    "AWS_S3_BUCKET": "your-bucket-name",
}

# If using temporary credentials (SSO/federated), add the session token:
# AWS_CREDENTIALS["AWS_SESSION_TOKEN"] = "your-session-token"

print(f"Using bucket: {AWS_CREDENTIALS['AWS_S3_BUCKET']}")

## Submit RayJob (managed Ray cluster + Kueue local queue)

In [ ]:
managed = ManagedClusterConfig(
    num_workers=2,
    head_cpu_requests=2,
    head_cpu_limits=4,
    head_memory_requests=4,
    head_memory_limits=8,
    worker_cpu_requests=2,
    worker_cpu_limits=4,
    worker_memory_requests=4,
    worker_memory_limits=8,
)

job = RayJob(
    job_name=JOB_NAME,
    entrypoint="python train_with_checkpoints.py",
    cluster_config=managed,
    namespace=NAMESPACE,
    local_queue=LOCAL_QUEUE,
    runtime_env={
        "working_dir": "./",
        "pip": ["torch", "torchvision", "s3fs", "pyarrow"],
        "env_vars": {
            **AWS_CREDENTIALS,
            "RAY_TRAIN_WORKER_GROUP_START_TIMEOUT_S": "120",  # Allow time for worker scheduling
        },
    },
)

job.submit()
print(
    "RayJob submitted. If status stays Suspended briefly, Kueue may still be admitting the Workload."
)
print(f"RayCluster name (when assigned): {job.cluster_name}")
print("Watch logs for: NO CHECKPOINT FOUND - Starting fresh")

## Monitor job progress (status + Ray dashboard Jobs)

Poll `job.status()`, then open the **Ray dashboard** URL from the RayCluster created by your RayJob. Use **Jobs** in the dashboard for live driver logs (epochs, checkpoint messages).

In [ ]:
print(job.status())
# KubeRay assigns a generated name — not job.cluster_name (the template).
cluster = None
for _ in range(36):
    status_data = job._api.get_job_status(name=job.name, k8s_namespace=NAMESPACE)
    ray_cluster_name = (status_data or {}).get("rayClusterName")
    if ray_cluster_name:
        cluster = get_cluster(ray_cluster_name, namespace=NAMESPACE, verify_tls=False)
        if cluster is not None:
            break
    time.sleep(5)
if cluster is None:
    raise RuntimeError(
        "RayCluster not ready — check RayJob / Workload admission and operator logs."
    )
print(f"Ray Dashboard (open in browser): {cluster.cluster_dashboard_uri()}")
print("In the dashboard, open Jobs and stream logs for the training driver.")
print(
    "Wait for at least one full epoch and a checkpoint to S3 before running the suspend cell."
)

## Suspend RayJob (checkpoint demo)

After logs show at least **one epoch** and a checkpoint written to **S3**, suspend the RayJob. This is a **manual** suspend for the demo (distinct from Kueue holding the job until admission right after submit).

Use **Pause** in the OpenShift AI UI, or run the next cell (`job.stop()`).

In [ ]:
print("=" * 60)
print("SUSPENDING RayJob (checkpoint demo — not deleting the RayJob CR)")
print("Checkpoints remain in S3.")
print("=" * 60)

job.stop()
print("Stop requested; poll job.status() until the RayJob reports suspended / non-running.")

## Resume RayJob

Use **Resume** in the OpenShift AI UI, or run `job.resubmit()` in the next cell. When the RayCluster is back, confirm in the dashboard **Jobs** view: `RESUMING FROM CHECKPOINT - Starting at epoch N`.

In [ ]:
print("=" * 60)
print("RESUMING RayJob after suspend")
print("Watch for RESUMING FROM CHECKPOINT in dashboard Jobs logs")
print("=" * 60)

job.resubmit()
time.sleep(10)
print(job.status())

## Verify resume from checkpoint

In the Ray dashboard **Jobs** tab, look for:

```
RESUMING FROM CHECKPOINT - Starting at epoch N
Previous loss: X.XXXX
```

That confirms optimizer and progress were restored from S3 across the suspend/resume cycle.

In [ ]:
print(job.status())
try:
    cluster = get_cluster(job.cluster_name, namespace=NAMESPACE, verify_tls=False)
    print(f"Ray Dashboard: {cluster.cluster_dashboard_uri()}")
except Exception as e:
    print(f"Could not resolve cluster yet: {e}")
print("Check Jobs tab for: RESUMING FROM CHECKPOINT - Starting at epoch N")

## Cleanup

Delete the RayJob and tear down the RayCluster if it is still present.

In [ ]:
print("Cleaning up...")
cluster_name = job.cluster_name
try:
    job.delete()
except Exception:
    pass

try:
    c = get_cluster(cluster_name, namespace=NAMESPACE, verify_tls=False)
    c.down()
except Exception:
    pass

print("Cleanup attempted (RayJob delete; cluster.down if RayCluster still exists).")

In [ ]:
# No explicit logout needed - authentication is managed automatically by kube-authkit